# 00 · Preparar o Drive para a coleta nova de 200 patentes

**Etapa 1 · CPU.** Esta execução confere a versão do projeto, cria os diretórios do piloto e inventaria os insumos. O modelo de extração previsto é o Qwen3-8B com o adaptador do próprio piloto.

1. Atualize em **Meu Drive** a cópia completa da pasta local do projeto. Guarde o nome e a estrutura de diretórios. Abra este caderno no Colab e selecione **CPU**. Execute as células em ordem e autorize a montagem do Drive.
2. Comece com as pastas do piloto vazias. O PDF de referência acompanha a entrega. Quando você fizer a nova coleta, preencha o registro de proveniência e coloque nele a consulta, os filtros, a data e a identificação da fonte conforme o notebook 02.
3. O manifesto vem com caminhos reservados para a exportação e as saídas futuras. O relatório informará quais arquivos existem. Uma assinatura SHA-256 identifica a cópia encontrada; para comprovar a origem, confira também a proveniência da nova consulta e os termos de exportação.

## 1. Montar o Drive e selecionar a pasta

Se usou uma subpasta de Meu Drive ou um drive compartilhado, altere `PASTA_DRIVE` antes de executar. O caminho começa por `/content/drive/`. O caminho do Desktop funciona somente no Windows. O relatório usa UTC para permitir comparação entre execuções.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
PASTA_DRIVE = "/content/drive/MyDrive/Replicação LLM EIDT IC"
ponto_de_montagem = Path("/content/drive").resolve(strict=True)
raiz = Path(PASTA_DRIVE).expanduser().resolve(strict=True)
if raiz == ponto_de_montagem or not raiz.is_relative_to(ponto_de_montagem) or not raiz.is_dir():
    raise ValueError("Selecione a pasta do projeto dentro do Drive montado.")
print("Pasta selecionada:", raiz)

## 2. Conferir a versão do piloto

O índice `SHA256SUMS.txt` da entrega local deve acompanhar o projeto no Drive. O caderno compara a configuração, o catálogo do piloto, os requisitos e o programa de seleção com o índice. Recopie o arquivo indicado e o índice da mesma entrega se a conferência falhar.

In [ ]:
import hashlib
import json
import re

def sha256_arquivo(caminho):
    resumo = hashlib.sha256()
    with caminho.open("rb") as fluxo:
        for bloco in iter(lambda: fluxo.read(1024 * 1024), b""):
            resumo.update(bloco)
    return resumo.hexdigest()

alvos = ("requirements.txt", "config/experimento.json", "config/insumos_piloto_200_exemplo.csv", "scripts/selecionar_piloto.py")
indice = raiz / "SHA256SUMS.txt"
if not indice.is_file():
    raise FileNotFoundError("Envie SHA256SUMS.txt da pasta do Desktop para a raiz do projeto no Drive.")
assinaturas = {}
for linha in indice.read_text(encoding="utf-8-sig").splitlines():
    match = re.fullmatch(r"([0-9a-fA-F]{64})  (.+)", linha)
    if match and match.group(2) in alvos:
        chave = match.group(2)
        if chave in assinaturas:
            raise ValueError(f"Entrada duplicada no índice para {chave}. Recopie o índice do Desktop.")
        assinaturas[chave] = match.group(1).lower()

for nome in alvos:
    caminho = raiz / nome
    if nome not in assinaturas or not caminho.is_file() or sha256_arquivo(caminho) != assinaturas[nome]:
        raise ValueError(f"Falha de integridade em {nome}. Recopie o arquivo e o índice da mesma entrega.")
    print("Conferido:", nome)

pasta_config = raiz / "config"
configuracao = json.loads((pasta_config / "experimento.json").read_text(encoding="utf-8-sig"))
if configuracao["protocolo"]["id"] != "llm-eidt-cn-piloto-200-8b":
    raise ValueError("Configuração de outro protocolo. Recopie a entrega do piloto de 200.")
if configuracao["modelos"]["qwen"]["id"] != "Qwen/Qwen3-8B" or configuracao["piloto"]["amostragem"]["total"] != 200:
    raise ValueError("Confira a versão da configuração do Qwen3-8B e da amostra.")
for chave in ("qwen", "embeddings"):
    modelo = configuracao["modelos"][chave]
    print(chave, modelo["id"], "· revisão", modelo["revisao"])

for nome in ("dados/brutos/piloto_200", "dados/referencia/piloto_200", "dados/intermediarios/piloto_200",
             "dados/processados/piloto_200", "modelos/piloto_200", "resultados/etapa_1/piloto_200"):
    (raiz / nome).mkdir(parents=True, exist_ok=True)
print("Amostra planejada:", configuracao["piloto"]["amostragem"]["particoes"])

## 3. Registrar insumos do piloto

A primeira execução disponibiliza `config/insumos_piloto_200.csv` a partir de `config/insumos_piloto_200_exemplo.csv`. Edite a cópia de trabalho em UTF-8, com vírgulas, preservando o cabeçalho e os IDs.

- Indique em `acesso_declarado` a situação da fonte ou da entrega: `nao_verificado`, `solicitado`, `disponivel` ou `indisponivel`.
- Quatro caminhos já indicam os locais de trabalho do notebook 02. A entrada é `dados/brutos/piloto_200/exportacao.csv` com `proveniencia.json` ao lado. A saída é `dados/processados/piloto_200/piloto_200.csv`, com `auditoria.json` na mesma pasta.
- Registre a anotação humana independente em um caminho sob `dados/referencia/piloto_200/` quando ela existir. Use uma linha por item. Inclua o SHA-256 de referência quando este estiver disponível.
- Antes de treinar, confira manualmente fonte, datas, autorização de uso, IDs de família e cobertura do CPC principal. Marque os acessos verificados no manifesto editável. O notebook preserva esse manifesto nas execuções seguintes.

In [ ]:
from shutil import copyfileobj

modelo_manifesto = pasta_config / "insumos_piloto_200_exemplo.csv"
manifesto = pasta_config / "insumos_piloto_200.csv"
try:
    with modelo_manifesto.open("rb") as fonte, manifesto.open("xb") as destino:
        copyfileobj(fonte, destino)
    print("Criado config/insumos_piloto_200.csv. Preencha os dados disponíveis antes da próxima célula.")
except FileExistsError:
    print("Usando o manifesto editável existente em config/insumos_piloto_200.csv.")

## 4. Auditar o manifesto e salvar o relatório

O caderno valida o formato do manifesto antes de abrir os insumos. Se apontar um erro, ajuste a linha indicada e execute novamente esta seção. O relatório registra itens sem arquivo, sem referência de hash ou com divergências para que possam ser investigados. Um estado pendente é uma conclusão válida para a Etapa 1.

In [ ]:
import csv
from datetime import datetime, timezone
from pathlib import PurePosixPath
from collections import Counter

CALCULAR_HASH_SEM_REFERENCIA = True
pastas_permitidas = {"dados/brutos/piloto_200", "dados/referencia/piloto_200", "dados/processados/piloto_200"}
permissoes = {"nao_verificado", "solicitado", "disponivel", "indisponivel"}

def ler_manifesto(caminho):
    with caminho.open(encoding="utf-8-sig", newline="") as arquivo:
        leitor = csv.DictReader(arquivo)
        colunas = ["id", "material", "uso", "acesso_declarado", "arquivo_relativo", "sha256_esperado"]
        if leitor.fieldnames != colunas:
            raise ValueError("Cabeçalho inválido. Compare com config/insumos_piloto_200_exemplo.csv e use vírgulas.")
        registros = {}
        for linha, item in enumerate(leitor, 2):
            if None in item or None in item.values():
                raise ValueError(f"Linha {linha} com número incorreto de campos. Confira as aspas e vírgulas.")
            item = {chave: valor.strip() for chave, valor in item.items()}
            ident = item["id"]
            if not re.fullmatch(r"[a-z][a-z0-9_]{0,63}", ident) or ident in registros:
                raise ValueError(f"ID ausente, inválido ou duplicado na linha {linha}.")
            if item["acesso_declarado"] not in permissoes:
                raise ValueError(f"acesso_declarado inválido em {ident}. Consulte a seção 3.")
            registros[ident] = item
    return registros

def caminho_autorizado(texto):
    partes = PurePosixPath(texto)
    if (partes.is_absolute() or partes.as_posix() != texto or ".." in partes.parts
        or "\\" in texto or ":" in texto or any(ord(c) < 32 for c in texto)):
        raise ValueError("Use caminho relativo com /, sem .., letras de unidade ou controles.")
    caminho = (raiz / partes).resolve()
    if caminho != (raiz / "dados/referencia/artigo_llm_eidt_v1.pdf") and not any(
        (raiz / pasta) in caminho.parents for pasta in pastas_permitidas
    ):
        raise ValueError("Use o PDF da referência ou um arquivo nos diretórios do piloto de dados brutos, processados ou de referência.")
    return caminho

def auditar_item(item):
    resultado = {"id": item["id"], "acesso_declarado": item["acesso_declarado"]}
    relativo = item["arquivo_relativo"]
    esperado = item["sha256_esperado"].lower()
    if esperado and (not relativo or not re.fullmatch(r"[0-9a-f]{64}", esperado)):
        raise ValueError(f"Revise arquivo_relativo e sha256_esperado de {item['id']}.")
    if not relativo:
        return {**resultado, "estado_arquivo": "nao_informado"}
    try:
        caminho = caminho_autorizado(relativo)
    except ValueError as erro:
        raise ValueError(f"{item['id']}: {erro}") from erro
    if not caminho.is_file():
        return {**resultado, "estado_arquivo": "arquivo_nao_localizado"}
    resultado["bytes"] = caminho.stat().st_size
    if resultado["bytes"] == 0:
        return {**resultado, "estado_arquivo": "arquivo_vazio"}
    if esperado or CALCULAR_HASH_SEM_REFERENCIA:
        resultado["sha256"] = sha256_arquivo(caminho)
        if esperado:
            resultado["sha256_esperado"] = esperado
            estado = "hash_confere" if resultado["sha256"] == esperado else "hash_divergente"
        else:
            estado = "hash_calculado_sem_referencia"
    else:
        estado = "presente_sem_hash"
    return {**resultado, "estado_arquivo": estado}

catalogo = ler_manifesto(modelo_manifesto)
declaracoes = ler_manifesto(manifesto)
if declaracoes.keys() != catalogo.keys():
    raise ValueError("Os IDs do manifesto diferem do modelo. Preserve todos os itens de config/insumos_piloto_200_exemplo.csv.")
itens = [auditar_item(declaracoes[ident]) for ident in catalogo]
agora = datetime.now(timezone.utc)
relatorio = {
    "versao_esquema": 2,
    "protocolo_id": configuracao["protocolo"]["id"],
    "modelo_extracao": configuracao["modelos"]["qwen"]["id"],
    "amostra_prevista": configuracao["piloto"]["amostragem"]["total"],
    "gerado_em_utc": agora.isoformat(),
    "configuracoes_sha256": assinaturas,
    "manifesto_sha256": sha256_arquivo(manifesto),
    "hash_sem_referencia_habilitado": CALCULAR_HASH_SEM_REFERENCIA,
    "escopo": "Auditoria da infraestrutura e dos insumos da coleta nova. Presença e hash exigem conferência de origem com o registro da consulta.",
    "contagens_por_estado": dict(Counter(item["estado_arquivo"] for item in itens)),
    "itens": itens,
}
nome_saida = "insumos_" + agora.strftime("%Y%m%d_%H%M%S_%fZ") + ".json"
saida = raiz / "resultados/etapa_1/piloto_200" / nome_saida
with saida.open("x", encoding="utf-8") as arquivo:
    json.dump(relatorio, arquivo, ensure_ascii=False, indent=2)
    arquivo.write("\n")
print("Relatório:", saida.relative_to(raiz))
for item in itens:
    print(item["id"], "· acesso", item["acesso_declarado"], "· arquivo", item["estado_arquivo"])

## 5. Ler o relatório

Na primeira execução, o resultado esperado é `hash_confere` para o artigo. As entradas e saídas do piloto aparecem como `arquivo_nao_localizado` até a coleta e a seleção. A anotação permanece `nao_informado` até ser registrada. Se uma assinatura divergir, confira a versão e a origem do arquivo. Os relatórios datados desta auditoria ficam em `resultados/etapa_1/piloto_200/`.

Você já pode verificar o ambiente de GPU no notebook 01. Para preparar as 200 patentes, siga o notebook 02 com a exportação nova. Depois, execute novamente esta auditoria e guarde os relatórios antes e depois da coleta. Para levar os resultados ao Desktop, baixe-os do Drive ou use a sincronização da pasta.